# LLM Hawkish/Dovish Classification — 8 models × 5 prompts

Zero-/few-shot stance classification of FOMC sentences (labels: 0 = dovish, 1 = hawkish, 2 = neutral)
across **2× Anthropic, 2× OpenAI, 2× Mistral, 2× DeepSeek**, with 5 prompt variants per model.

**Reproducibility:** all sampling settings are pinned (see *Reproducibility settings* below), every raw
model response is saved to `results/`, and each run writes a `run_manifest.json` recording model IDs,
prompt hashes, data-file checksums, package versions, and the random seed. Run order: edit
**Configuration**, run all cells top-to-bottom, pilot with `LIMIT = 25` before the full run.

## 0. Data files — set the explicit paths

The three dataset files are referenced by **explicit full paths**, declared once in the
**Configuration** cell below — edit those three lines to wherever the files live on your
machine; they do **not** need to sit next to the notebook:

```python
TRAIN_PATH = Path(r"C:\Users\yourname\Documents\study\train.jsonl")
VAL_PATH   = Path(r"C:\Users\yourname\Documents\study\val.jsonl")
TEST_PATH  = Path(r"C:\Users\yourname\Documents\study\test.jsonl")
```

Notes on writing the paths: on **Windows** keep the `r"..."` raw-string form (or use forward
slashes, `Path("C:/Users/...")`) so backslashes aren't treated as escapes; on **macOS/Linux**
it's simply `Path("/Users/yourname/study/train.jsonl")`; on **Colab**, upload via the Files
pane and use `Path("/content/train.jsonl")` etc. Each file must be JSON-Lines with one
`{"text": ..., "label": 0|1|2}` object per line.

The *Preflight* cell verifies every configured path exists (stopping with a clear error naming
the missing file if not) and records each file's SHA-256 checksum and label distribution in
`run_manifest.json`. Outputs go to `results/` next to the notebook — make `OUTDIR` an explicit
absolute path too if you prefer.

## Packages

Only two SDKs are needed — `openai` also drives Mistral and DeepSeek via their
OpenAI-compatible endpoints. For the paper, record the exact versions you used
(the Preflight cell logs them into `run_manifest.json` automatically):

```
pip install anthropic openai
```

In [1]:
import json
import os
import random
import re
import sys
import threading
import time
import zlib
import hashlib
import platform
from datetime import datetime, timezone
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

# Global seed: governs few-shot ordering, dry-run fakes, and is sent to every
# API that accepts a seed parameter (see Reproducibility settings).
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

## Label Assignment

The three prompts are generated from / checked against this dict, so it is the **single
source of truth** for label semantics. This follows the standard FOMC-stance convention
(Shah et al., 2023) — verified against the dataset before the full run.

In [2]:
LABEL_MAPPING = {
    0: "Dovish --> Leans towards lower interest rates, more accommodative monetary policy.",
    1: "Hawkish --> Leans towards higher interest rates, tighter monetary policy.",
    2: "Neutral --> Neither strongly dovish nor strongly hawkish, balanced stance.",
}

## LLM Models --> Anthropic, OpenAI, Mistral, Deepseek

**Reproducibility note:** `-latest` aliases move silently when providers ship new versions.
Before the run you report in the paper, replace them with **dated snapshots** (e.g.
`claude-haiku-4-5-20251001`, `mistral-large-2411`, a dated `gpt-5.1-...`) from each
provider's model page, so the exact weights are pinned. Prices ($/1M tokens) are used
for realized-cost tracking and should be re-checked against the pricing pages.

In [3]:
MODELS = {
    "claude-sonnet": {
        "provider": "anthropic", "model_id": "claude-sonnet-4-6",
        "price_in": 3.00, "price_out": 15.00,
    },
    "claude-haiku": {
        "provider": "anthropic", "model_id": "claude-haiku-4-5",
        "price_in": 1.00, "price_out": 5.00,
    },
    "gpt-5.1": {
        "provider": "openai", "model_id": "gpt-5.1",
        "price_in": 1.25, "price_out": 10.00,
        # 'none' switches off reasoning for GPT-5.1; if the API rejects it,
        # fall back to "minimal". Critical for cost (see study budget).
        "extra": {"reasoning_effort": "none"},
        "no_temperature": True,          # gpt-5 family rejects temperature != 1
        "use_max_completion_tokens": True,
        "max_out": 16,
    },
    "gpt-5-mini": {
        "provider": "openai", "model_id": "gpt-5-mini",
        "price_in": 0.25, "price_out": 2.00,
        "extra": {"reasoning_effort": "minimal"},   # lowest available for mini
        "no_temperature": True,
        "use_max_completion_tokens": True,
        "max_out": 16,
    },
    "mistral-large": {
        "provider": "mistral", "model_id": "mistral-large-latest",
        "price_in": 2.00, "price_out": 6.00,
    },
    "mistral-small": {
        "provider": "mistral", "model_id": "mistral-small-latest",
        "price_in": 0.10, "price_out": 0.30,
    },
    "deepseek-chat": {
        "provider": "deepseek", "model_id": "deepseek-chat",
        "price_in": 0.28, "price_out": 0.42,
    },
    "deepseek-reasoner": {
        "provider": "deepseek", "model_id": "deepseek-reasoner",
        "price_in": 0.28, "price_out": 0.42,
        # max_tokens caps the FINAL answer only; CoT is separate but billed
        # (it shows up in usage.completion_tokens, so costs stay accurate).
        "no_temperature": True,
        "max_out": 32,
    },
}

## Provider API keys — secure setup

In [4]:
PROVIDER_ENV = {                 # provider -> name of the environment variable
    "anthropic": "ANTHROPIC_API_KEY",
    "openai":    "OPENAI_API_KEY",
    "mistral":   "MISTRAL_API_KEY",
    "deepseek":  "Deepseek_API_KEY",
}
PROVIDER_BASE_URL = {            # OpenAI-compatible endpoints
    "openai": None,              # SDK default
    "mistral": "https://api.mistral.ai/v1",
    "deepseek": "https://api.deepseek.com",
}

import getpass

def _ensure_keys(providers):
    """Prompt (hidden input) for any missing keys among the given providers."""
    for prov in providers:
        env = PROVIDER_ENV[prov]
        if not os.environ.get(env):
            os.environ[env] = getpass.getpass(f"Enter {env} (input hidden, kept in memory only): ")

## System Prompts

Five fixed variants (`p1_analyst_direct` … `p5_analyst_fewshot`). Treat them as **frozen** once
the study starts: the Preflight cell records a SHA-256 hash of this dict in `run_manifest.json`,
so any later edit is detectable. `{sentence}` and `{examples}` are filled at runtime; the few-shot
examples for p5 are picked deterministically (first occurrence of each class in `FEWSHOT_FILE`).

In [5]:
PROMPTS = {
    # 1. Direct & Concise (Anchored Analyst)
    "p1_analyst_direct": {
        "system": (
            "You are an expert monetary policy analyst. Your job is to classify central bank "
            "communications into monetary policy stances using specific domain guidelines. "
            "Output a single digit corresponding to the correct label and nothing else."
        ),
        "user": (
            "Classify the following sentence based on standard central bank communication rules:\n"
            "- Dovish (0): Decreasing inflation, rising unemployment, low growth projections, dollar appreciation, falling energy/house prices, decreasing yields/spreads, or focus on 'accommodative' / 'maximum employment'.\n"
            "- Hawkish (1): Increasing inflation, falling unemployment, economic output above potential, dollar depreciation, rising energy/house prices, increasing yields/TIPS, or focus on 'price stability' / 'sustained growth'.\n"
            "- Neutral (2): Unchanged/sustained unemployment or growth, reference to foreign nation policies, or neutral phrasing like 'mixed', 'moderate', or 'reaffirmed'.\n\n"
            "Sentence: \"{sentence}\"\n\n"
            "Label digit (0, 1, or 2):"
        ),
    },

    # 2. Comprehensive Guidelines (Detailed Rubric)
    "p2_analyst_rubric": {
        "system": (
            "You are a Senior Monetary Policy Analyst at a central bank specializing in FOMC statement analysis. "
            "You strictly adhere to formal annotation rubrics to label central bank sentiment. "
            "Respond with ONLY a single digit: 0, 1, or 2."
        ),
        "user": (
            "Analyze the policy statement below using the following Annotation Criteria:\n\n"
            "1. DOVISH (0):\n"
            "   - Economic Status: Decreasing inflation, increasing unemployment, low projected growth.\n"
            "   - Prices & Dollar: Dollar appreciates, falling oil/energy/house prices.\n"
            "   - Markets & Fed: Expecting disinflation/subpar inflation, narrowing treasury spreads, decreasing treasury yields, reduction of bank reserves.\n"
            "   - Money/Labor: Low money supply, M2 increase, higher loan demand, increasing productivity.\n"
            "   - Phrasing: Focus on 'accommodative' policy or 'maximum employment'.\n\n"
            "2. HAWKISH (1):\n"
            "   - Economic Status: Increasing inflation, decreasing unemployment, high projected growth, output above potential, falling economic slack.\n"
            "   - Prices & Dollar: Dollar depreciates, rising oil/energy/house prices.\n"
            "   - Markets & Fed: Expecting high inflation, widening treasury spreads, increasing treasury yields/TIPS value, increasing bank reserves.\n"
            "   - Money/Labor: High money supply, increased demand for goods, low demand for loans, decreasing productivity.\n"
            "   - Phrasing: Focus on 'price stability' or 'sustained growth'.\n\n"
            "3. NEUTRAL (2):\n"
            "   - Status/Foreign: Unemployment or growth rate is unchanged/maintained; references to foreign nation trade/economic policies.\n"
            "   - Phrasing: Uses balanced words like 'mixed', 'moderate', or 'reaffirmed'.\n\n"
            "Sentence: \"{sentence}\"\n\n"
            "Digit:"
        ),
    },

    # 3. Rule-Based Chain of Constraint
    "p3_analyst_rules": {
        "system": (
            "You are a central bank research assistant. You map central bank statements to monetary policy stances. "
            "Output only the digit matching the correct stance (0, 1, or 2)."
        ),
        "user": (
            "Task: Classify the stance of the sentence below using central bank annotation rules.\n\n"
            "Annotation Rules:\n"
            "- Classify as 0 (Dovish) if text indicates falling inflation, rising unemployment, economic slack, decreasing yields/energy prices, or mentions 'accommodative'.\n"
            "- Classify as 1 (Hawkish) if text indicates rising inflation, falling unemployment, output above potential, rising yields/energy prices, or mentions 'price stability'.\n"
            "- Classify as 2 (Neutral) if economic variables are maintained/unchanged, refers to foreign trade policies, or uses balanced terms ('mixed', 'moderate', 'reaffirmed').\n"
            "- Stance Priority: Judge only explicit stance expressed in the statement. Prefer 2 (Neutral) if data is reported without clear directional bias.\n\n"
            "Sentence: \"{sentence}\"\n"
            "Label (0, 1, or 2):"
        ),
    },

    # 4. JSON API (Central Bank NLP Pipeline)
    "p4_analyst_json": {
        "system": (
            "You are an automated central bank sentiment classification engine. "
            "Evaluate text against monetary policy annotation criteria and return raw JSON only."
        ),
        "user": (
            "Classify this policy text according to FOMC stance criteria:\n"
            "- 0 (Dovish): Lower rates/inflation, higher unemployment, dollar appreciation, falling yields/energy prices, focus on accommodation.\n"
            "- 1 (Hawkish): Higher rates/inflation, lower unemployment, dollar depreciation, rising yields/energy prices, focus on price stability.\n"
            "- 2 (Neutral): Sustained/unchanged indicators, foreign policy context, or neutral terms ('moderate', 'mixed', 'reaffirmed').\n\n"
            "Sentence: \"{sentence}\"\n\n"
            "Respond ONLY with this format: {\"label\": <digit>}"
        ),
    },

    # 5. Few-Shot (Domain Expert Guided)
    "p5_analyst_fewshot": {
        "system": (
            "You are a senior monetary policy analyst carefully annotating central bank communications. "
            "Reply with a single digit (0, 1, or 2) only."
        ),
        "user": (
            "Use the standard annotation guidelines (Dovish=0: disinflation/slack/accommodation, "
            "Hawkish=1: inflation/tight labor/price stability, Neutral=2: unchanged metrics/foreign policy/mixed terms).\n\n"
            "Examples:\n{examples}\n\n"
            "Now classify this statement:\n"
            "Sentence: \"{sentence}\"\n"
            "Label:"
        ),
    },
}

In [6]:
def _labels_block():
    """Render label definitions from LABEL_MAPPING (single source of truth)."""
    return "\n".join(f"{k} = {v}" for k, v in LABEL_MAPPING.items())

def build_fewshot_examples(path):
    """Pick the first example of each class from a JSONL file (e.g. train.jsonl)."""
    picked = {}
    if path and Path(path).exists():
        with open(path) as fh:
            for line in fh:
                if not line.strip():
                    continue
                r = json.loads(line)
                if r["label"] in LABEL_MAPPING and r["label"] not in picked:
                    picked[r["label"]] = r["text"]
                if len(picked) == len(LABEL_MAPPING):
                    break
    return "\n".join(f'Sentence: "{t}"\nLabel: {k}\n' for k, t in sorted(picked.items()))

def render_prompt(prompt_id, sentence, examples):
    p = PROMPTS[prompt_id]
    user = (p["user"]
            .replace("{labels}", _labels_block())
            .replace("{examples}", examples)
            .replace("{sentence}", sentence.replace('"', "'")))
    return p["system"], user

## Reproducibility settings

Everything that can be pinned, is pinned — set in code in the next cells and summarized here:

| Setting | Value | Applies to |
|---|---|---|
| `temperature` | **0** | Claude Sonnet/Haiku, Mistral Large/Small, DeepSeek-chat |
| `temperature` | *not sent* (API fixed default) | GPT-5.1, GPT-5-mini — the GPT-5 family rejects non-default temperature; determinism is approximated via `seed` + `reasoning_effort` |
| `temperature` | *ignored by API* | DeepSeek-reasoner (documented behaviour) |
| `seed` | **42** (`RANDOM_SEED`) | OpenAI models (`seed=42`, best-effort determinism), Mistral (`random_seed=42`) — Anthropic and DeepSeek expose no seed parameter |
| `top_p` / other samplers | provider defaults, untouched | all — mixing `top_p` with `temperature=0` is discouraged and adds nothing |
| `reasoning_effort` | `"none"` (GPT-5.1) / `"minimal"` (GPT-5-mini) | pins reasoning off — also the difference between ~$0.72 and ~$7.60 on the test run |
| max output tokens | 8 (16 for GPT-5.x, 32 for DeepSeek-reasoner's final answer) | all — forces label-only outputs |
| Python RNG | `random.seed(42)`; dry-run fakes use `zlib.crc32` (stable across sessions, unlike `hash()`) | pipeline itself |
| Few-shot examples | first sentence of each class in `FEWSHOT_FILE`, in label order | p5 prompt |
| Model versions | pin dated snapshots for the paper run | all |

**Honest caveat for the methods section:** `temperature=0` plus a seed makes outputs *highly stable*
but does **not** guarantee bit-identical responses — providers batch requests on changing hardware,
and hosted models can be silently updated. That is why true reproducibility here rests on:
(1) pinned dated model snapshots, (2) `run_manifest.json` (seed, model IDs, prompt hash, data
checksums, package versions, timestamp), and (3) the fact that **every raw response is stored** in
`results/*.jsonl`, so all metrics can be recomputed from disk without re-calling any API.
Report macro-F1 as mean ± SD across the five prompts — the summary cell prints exactly that.

## API Plumbing

Rate limiting per provider, exponential-backoff retries, and the API calls with the sampling settings above.

In [7]:
class RateLimiter:
    """Simple global throttle: at most `rpm` requests/minute per provider."""
    def __init__(self, rpm):
        self.interval = 60.0 / max(rpm, 1)
        self.lock = threading.Lock()
        self.next_ok = 0.0
    def wait(self):
        with self.lock:
            now = time.time()
            wait_for = max(0.0, self.next_ok - now)
            self.next_ok = max(now, self.next_ok) + self.interval
        if wait_for > 0:
            time.sleep(wait_for)
 
_clients = {}
def get_client(provider):
    if provider in _clients:
        return _clients[provider]
    key = os.environ.get(PROVIDER_ENV[provider])
    if not key:
        raise RuntimeError(f"Missing {PROVIDER_ENV[provider]} -- set the environment "
                           f"variable or run _ensure_keys([...]) in the API-keys cell.")
    if provider == "anthropic":
        import anthropic
        _clients[provider] = anthropic.Anthropic(api_key=key)
    else:
        from openai import OpenAI
        base = PROVIDER_BASE_URL[provider]
        _clients[provider] = OpenAI(api_key=key, base_url=base) if base else OpenAI(api_key=key)
    return _clients[provider]

def call_model(mkey, system, user, dry_run=False):
    """Returns (raw_text, in_tokens, out_tokens). Raises on final failure."""
    cfg = MODELS[mkey]
    if dry_run:  # deterministic fake: lets you test the pipeline with no keys
        fake = str(random.Random(zlib.crc32(user.encode())).choice(list(LABEL_MAPPING)))
        return fake, int(len(system + user) / 4) + 7, 5

    provider, max_out = cfg["provider"], cfg.get("max_out", 8)
    last_err = None
    for attempt in range(6):
        try:
            if provider == "anthropic":
                r = get_client(provider).messages.create(
                    model=cfg["model_id"], max_tokens=max_out, temperature=0,
                    system=system, messages=[{"role": "user", "content": user}],
                )
                return (r.content[0].text if r.content else ""), r.usage.input_tokens, r.usage.output_tokens
            kw = dict(model=cfg["model_id"],
                      messages=[{"role": "system", "content": system},
                                {"role": "user", "content": user}])
            if cfg.get("use_max_completion_tokens"):
                kw["max_completion_tokens"] = max_out
            else:
                kw["max_tokens"] = max_out
            if not cfg.get("no_temperature"):
                kw["temperature"] = 0
            if provider == "openai":
                kw["seed"] = RANDOM_SEED          # best-effort determinism (OpenAI)
            elif provider == "mistral":
                kw["random_seed"] = RANDOM_SEED   # Mistral's seed parameter
            kw.update(cfg.get("extra", {}))
            r = get_client(provider).chat.completions.create(**kw)
            txt = r.choices[0].message.content or ""
            # note: for deepseek-reasoner, usage.completion_tokens includes CoT
            return txt, r.usage.prompt_tokens, r.usage.completion_tokens
        except Exception as e:  # 429s, 5xx, transient network -- back off & retry
            last_err = e
            time.sleep(min(2 ** attempt * 2, 60) + random.random())
    raise RuntimeError(f"{mkey}: failed after retries: {last_err}")

## Parsing Metrics

Robust label extraction (JSON form, bare digit, then word fallback) and manual macro-F1 — no sklearn dependency.

In [8]:
_valid = "".join(str(k) for k in LABEL_MAPPING)

def parse_label(raw):
    if not raw:
        return None
    m = re.search(r'"label"\s*:\s*([%s])' % _valid, raw)       # JSON form
    if m:
        return int(m.group(1))
    m = re.search(r"\b([%s])\b" % _valid, raw)                 # bare digit
    if m:
        return int(m.group(1))
    low = raw.lower()                                          # word fallback
    for k, v in LABEL_MAPPING.items():
        if v.split()[0].lower() in low:
            return k
    return None

def macro_f1(golds, preds):
    scores = []
    for c in LABEL_MAPPING:
        tp = sum(1 for g, p in zip(golds, preds) if g == c and p == c)
        fp = sum(1 for g, p in zip(golds, preds) if g != c and p == c)
        fn = sum(1 for g, p in zip(golds, preds) if g == c and p != c)
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        scores.append(2 * prec * rec / (prec + rec) if prec + rec else 0.0)
    return sum(scores) / len(scores)

## Runner with resume + incremental writes

One JSONL per (model, prompt). Interrupted runs resume automatically — rerun the Run cell and finished calls are skipped.

In [9]:
def run_one(mkey, prompt_id, items, examples, outdir, limiter, workers, dry_run):
    outfile = outdir / f"results__{mkey}__{prompt_id}.jsonl"
    done = set()
    if outfile.exists():
        with open(outfile) as fh:
            for line in fh:
                try:
                    done.add(json.loads(line)["idx"])
                except Exception:
                    pass
    todo = [(i, it) for i, it in enumerate(items) if i not in done]
    if not todo:
        print(f"  {mkey} x {prompt_id}: already complete ({len(done)} rows)")
        return
    print(f"  {mkey} x {prompt_id}: {len(todo)} to do ({len(done)} resumed)")
    lock = threading.Lock()
    cfg = MODELS[mkey]
 
    def work(idx, item):
        if not dry_run:
            limiter.wait()
        system, user = render_prompt(prompt_id, item["text"], examples)
        try:
            raw, tin, tout = call_model(mkey, system, user, dry_run)
            err = None
        except Exception as e:
            raw, tin, tout, err = "", 0, 0, str(e)
        row = {
            "idx": idx, "model": mkey, "prompt": prompt_id,
            "gold": item["label"], "pred": parse_label(raw), "raw": raw.strip(),
            "in_tok": tin, "out_tok": tout,
            "cost": (tin * cfg["price_in"] + tout * cfg["price_out"]) / 1e6,
            "error": err,
        }
        with lock:
            with open(outfile, "a") as fh:
                fh.write(json.dumps(row) + "\n")
        return err
 
    n_err = 0
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futures = [ex.submit(work, i, it) for i, it in todo]
        for k, f in enumerate(as_completed(futures), 1):
            if f.result():
                n_err += 1
            if k % 50 == 0 or k == len(futures):
                print(f"    {mkey} x {prompt_id}: {k}/{len(futures)} ({n_err} errors)", flush=True)
 
def summarize(outdir):
    rows = []
    for f in sorted(outdir.glob("results__*.jsonl")):
        recs = [json.loads(l) for l in open(f) if l.strip()]
        if not recs:
            continue
        ok = [r for r in recs if r["pred"] is not None and not r["error"]]
        golds = [r["gold"] for r in ok]
        preds = [r["pred"] for r in ok]
        acc = sum(g == p for g, p in zip(golds, preds)) / len(ok) if ok else 0.0
        rows.append({
            "model": recs[0]["model"], "prompt": recs[0]["prompt"], "n": len(recs),
            "parsed": len(ok), "errors": sum(1 for r in recs if r["error"]),
            "accuracy": round(acc, 4),
            "macro_f1": round(macro_f1(golds, preds), 4) if ok else 0.0,
            "in_tok": sum(r["in_tok"] for r in recs),
            "out_tok": sum(r["out_tok"] for r in recs),
            "cost_usd": round(sum(r["cost"] for r in recs), 4),
        })
    if not rows:
        print("No results yet.")
        return
    import csv
    with open(outdir / "summary.csv", "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=rows[0].keys())
        w.writeheader(); w.writerows(rows)
    hdr = f"{'model':18s} {'prompt':12s} {'n':>5s} {'acc':>7s} {'macroF1':>8s} {'out_tok':>9s} {'cost$':>9s}"
    print("\n" + hdr + "\n" + "-" * len(hdr))
    for r in rows:
        print(f"{r['model']:18s} {r['prompt']:12s} {r['n']:5d} {r['accuracy']:7.3f} "
              f"{r['macro_f1']:8.3f} {r['out_tok']:9d} {r['cost_usd']:9.4f}")
    # per-model mean +/- sd across prompts (prompt-sensitivity, nice for the paper)
    by_model = defaultdict(list)
    for r in rows:
        by_model[r["model"]].append(r["macro_f1"])
    print("\nMacro-F1 across prompts (mean +/- sd):")
    for m, v in by_model.items():
        mean = sum(v) / len(v)
        sd = (sum((x - mean) ** 2 for x in v) / len(v)) ** 0.5
        print(f"  {m:18s} {mean:.3f} +/- {sd:.3f}  (k={len(v)})")
    total = sum(r["cost_usd"] for r in rows)
    print(f"\nTotal realized cost so far: ${total:.2f}")
    print(f"Summary written to {outdir/'summary.csv'}")

## Configuration — edit here, nothing else

Start with `LIMIT = 25` (pilot: verifies keys, parsing, and — via the `out_tok` column — that
GPT-5.1 reasoning is really off and how long DeepSeek-reasoner's CoT is). Then set `LIMIT = None`
for the full run. `DRY_RUN = True` exercises the whole pipeline with fake responses and no keys.

In [ ]:
# --- EXPLICIT DATA FILE PATHS: edit these three lines to your machine ---------
TRAIN_PATH = Path("/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Github Repos/Project/BdF-Project/data/train.jsonl")
VAL_PATH   = Path("/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Github Repos/Project/BdF-Project/data/val.jsonl")
TEST_PATH  = Path("/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Github Repos/Project/BdF-Project/data/test.jsonl")
# macOS/Linux example:  TRAIN_PATH = Path("/Users/yourname/study/train.jsonl")

FILES        = [TEST_PATH]             # which file(s) to classify this run,
                                       # e.g. [TRAIN_PATH, VAL_PATH, TEST_PATH] for everything
FEWSHOT_FILE = TRAIN_PATH              # source of the 3 few-shot examples (p5)

RUN_MODELS   = list(MODELS)            # all 8, or subset e.g. ["deepseek-reasoner", "gpt-5.1"]
RUN_PROMPTS  = list(PROMPTS)           # all 5

LIMIT        = None          # pilot on first N sentences; None = full run
WORKERS      = 4           # parallel requests per (model, prompt)
RPM          = 50          # max requests/minute per provider (raise if your tier allows)
OUTDIR       = Path("results")         # can also be absolute, e.g. Path(r"C:\Users\...\results")
DRY_RUN      = False       # True = no API calls, deterministic fake labels

## Preflight — data check + run manifest

Verifies the data files, prints label distributions, logs package versions, and writes
`results/run_manifest.json`. Attach that file to the paper's supplementary material —
together with `results/*.jsonl` it makes every number recomputable.

In [11]:
def _sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for b in iter(lambda: fh.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

OUTDIR.mkdir(exist_ok=True)
manifest = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "random_seed": RANDOM_SEED,
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "packages": {},
    "models": {k: MODELS[k]["model_id"] for k in RUN_MODELS},
    "prompts_sha256": hashlib.sha256(
        json.dumps({k: PROMPTS[k] for k in RUN_PROMPTS}, sort_keys=True).encode()).hexdigest(),
    "label_mapping": {str(k): v for k, v in LABEL_MAPPING.items()},
    "config": {"files": [str(f) for f in FILES], "fewshot_file": str(FEWSHOT_FILE),
               "limit": LIMIT, "workers": WORKERS, "rpm": RPM, "dry_run": DRY_RUN},
    "data_files": {},
}
for pkg in ("anthropic", "openai"):
    try:
        manifest["packages"][pkg] = __import__(pkg).__version__
    except Exception:
        manifest["packages"][pkg] = "NOT INSTALLED"

missing = [f for f in FILES + [FEWSHOT_FILE] if not Path(f).exists()]
if missing:
    raise FileNotFoundError(
        f"Data file(s) not found: {[str(m) for m in missing]} -- edit TRAIN_PATH / VAL_PATH / "
f"TEST_PATH in the Configuration cell to the full locations of your files (see section 0).")

for f in FILES:
    recs = [json.loads(l) for l in open(f) if l.strip()]
    dist = defaultdict(int)
    for r in recs:
        dist[r["label"]] += 1
    manifest["data_files"][str(f)] = {"n": len(recs), "sha256": _sha256(f),
                                      "label_dist": dict(sorted(dist.items()))}
    print(f"{f}: {len(recs)} sentences, labels {dict(sorted(dist.items()))}")

with open(OUTDIR / "run_manifest.json", "w") as fh:
    json.dump(manifest, fh, indent=2)
print(f"\npackages: {manifest['packages']}")
print(f"manifest written -> {OUTDIR/'run_manifest.json'}")

/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Github Repos/Project/BdF-Project/data/test.jsonl: 407 sentences, labels {0: 104, 1: 112, 2: 191}

packages: {'anthropic': '0.120.2', 'openai': '2.53.0'}
manifest written -> results/run_manifest.json


## Run

Safe to re-run at any time: completed (model, prompt, sentence) calls are skipped, so an
interrupted run continues where it stopped. The summary prints accuracy, macro-F1, token usage,
realized cost per combination, and macro-F1 mean ± SD across prompts per model.

In [ ]:
items = []
for f in FILES:
    with open(f) as fh:
        items += [json.loads(l) for l in fh if l.strip()]
if LIMIT:
    items = items[:LIMIT]
print(f"Loaded {len(items)} sentences from {[str(f) for f in FILES]}")

examples = build_fewshot_examples(FEWSHOT_FILE)
if any("{examples}" in PROMPTS[p]["user"] for p in RUN_PROMPTS) and not examples:
    print(f"WARNING: no few-shot examples found in {FEWSHOT_FILE}; few-shot prompt runs without examples.")

providers_needed = {MODELS[m]["provider"] for m in RUN_MODELS}
if not DRY_RUN:
    _ensure_keys(sorted(providers_needed))   # prompts only for missing keys
    for p in providers_needed:
        get_client(p)                        # fail fast on bad credentials

limiters = {p: RateLimiter(RPM) for p in PROVIDER_ENV}
n_calls = len(items) * len(RUN_MODELS) * len(RUN_PROMPTS)
print(f"Plan: {len(RUN_MODELS)} models x {len(RUN_PROMPTS)} prompts x {len(items)} sentences "
      f"= {n_calls:,} calls{'  [DRY RUN]' if DRY_RUN else ''}\n")

for mkey in RUN_MODELS:
    for pid in RUN_PROMPTS:
        run_one(mkey, pid, items, examples, OUTDIR,
                limiters[MODELS[mkey]["provider"]], WORKERS, DRY_RUN)

summarize(OUTDIR)

Loaded 25 sentences from ['/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Github Repos/Project/BdF-Project/data/test.jsonl']
Plan: 8 models x 5 prompts x 25 sentences = 1,000 calls

  claude-sonnet x p1_analyst_direct: 25 to do (0 resumed)
    claude-sonnet x p1_analyst_direct: 25/25 (0 errors)
  claude-sonnet x p2_analyst_rubric: 25 to do (0 resumed)
    claude-sonnet x p2_analyst_rubric: 25/25 (0 errors)
  claude-sonnet x p3_analyst_rules: 25 to do (0 resumed)
    claude-sonnet x p3_analyst_rules: 25/25 (0 errors)
  claude-sonnet x p4_analyst_json: 25 to do (0 resumed)
    claude-sonnet x p4_analyst_json: 25/25 (0 errors)
  claude-sonnet x p5_analyst_fewshot: 25 to do (0 resumed)
    claude-sonnet x p5_analyst_fewshot: 25/25 (0 errors)
  claude-haiku x p1_analyst_direct: 25 to do (0 resumed)
    claude-haiku x p1_analyst_direct: 25/25 (0 errors)
  claude-haiku x p2_analyst_rubric: 25 to do (0 resumed)
    claude-haiku x p2_analyst_rubric: 25/25 (0 errors)
  claude-haiku x p3_analyst_

In [ ]:
import csv, json
from collections import defaultdict

# rebuild the exact sentence list the run used (keep FILES/LIMIT as they were for the run)
sentences = []
for f in FILES:
    with open(f) as fh:
        sentences += [json.loads(l) for l in fh if l.strip()]
if LIMIT:
    sentences = sentences[:LIMIT]

name = {k: v.split()[0] for k, v in LABEL_MAPPING.items()}     # 0→Dovish, 1→Hawkish, 2→Neutral

rows, preds = [], defaultdict(dict)
for rf in sorted(OUTDIR.glob("results__*.jsonl")):
    for line in open(rf):
        r = json.loads(line)
        i = r["idx"]
        if i >= len(sentences): continue
        rows.append({"idx": i, "sentence": sentences[i]["text"],
                     "your_label": r["gold"], "your_label_name": name.get(r["gold"], "?"),
                     "model": r["model"], "prompt": r["prompt"],
                     "ai_label": r["pred"], "ai_label_name": name.get(r["pred"], "unparsed"),
                     "correct": int(r["pred"] == r["gold"]) if r["pred"] is not None else "",
                     "raw_model_reply": r["raw"]})
        preds[i][f'{r["model"]}__{r["prompt"]}'] = r["pred"]

rows.sort(key=lambda r: (r["idx"], r["model"], r["prompt"]))
with open(OUTDIR / "predictions_detailed.csv", "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)

combos = sorted({c for d in preds.values() for c in d})
with open(OUTDIR / "predictions_wide.csv", "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["idx", "sentence", "your_label", "your_label_name"] + combos)
    for i, s in enumerate(sentences):
        if i in preds:
            w.writerow([i, s["text"], s["label"], name.get(s["label"], "?")]
                       + [preds[i].get(c, "") for c in combos])

print(f"wrote predictions_detailed.csv ({len(rows)} rows) and predictions_wide.csv "
      f"({len(preds)} sentences × {len(combos)} model-prompt columns) in {OUTDIR}/")